# 评测数据整理与验收

现行标准：[S v2.2](../资料/评判标准.md)。评分来源：`项目2_AI评测分析.xlsx`中“评分与用量”的325条S分项及得分批注。查证数据须先按[维护步骤](../资料/复现说明.md)同步。

本Notebook只读当前交付文件，核对分值、依据同步和汇总；不调用参赛模型，不重做原始分析。旧Q/R代码与输出保存在[历史快照](../_实验系统/报告素材/历史_QR评测验收.ipynb)。

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP
from IPython.display import display
from openpyxl import load_workbook

# 从项目根目录或交付目录执行，避免依赖某台电脑的盘符。
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "评测数据处理.py").is_file())
sys.path.insert(0, str(ROOT))
import 评测数据处理 as pipeline
workbook = ROOT / "交付成果/项目2_AI评测分析.xlsx"
payload = ROOT / "交付成果/查证数据.js"
inputs = [workbook, payload, ROOT / "资料/评判标准.md"]
before = {p: hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}
book = load_workbook(workbook, data_only=True)
def frame(name):
    rows = list(book[name].values)
    return pd.DataFrame(rows[1:], columns=rows[0]).dropna(how="all")
scores = pipeline.read_current_scores()
detail = pd.DataFrame(v["评分原文"] for v in scores.values())
detail["维度"] = detail["条款"].str[0]
systems = ["GPT", "Grok", "DeepSeek", "GLM", "Qwen", "Gemini"]
all_systems = systems + ["Claude", "Kimi"]


## 1. 运行记录

读取正式Excel中的运行条件。模型名称与后续核定状态以此表为准；原生记录保留在SQLite和查证页中。

In [2]:
runs = frame("运行记录")
display(runs[["AI", "轮次", "运行状态", "运行工具", "模型"]])

,AI,轮次,运行状态,运行工具,模型
0,GPT,第一轮,已完成,Codex,gpt-6-astra
1,Grok,第一轮,已完成,Cursor,grok-4.6
2,DeepSeek,第一轮,已完成,Claude Code,deepseek-v4.1-flash
3,Gemini,第一轮,已完成,Antigravity,gemini-3.8-flash
4,Qwen,第一轮,已完成,Claude Code,qwen3.8-max-0902
5,GLM,第一轮,已完成,Claude Code,glm-5.3
6,DeepSeek,第二轮,已完成,Claude Code,deepseek-v4.1-flash
7,GPT,第二轮,已完成,Codex,gpt-6-astra
8,GLM,第二轮,已完成,Claude Code,glm-5.3
9,Kimi,第二轮,失败,Claude Code,kimi-k3


## 2. 参与与未评分状态

六系统两轮参与配对。Claude第一轮按接管前工作计分，第二轮未运行；Kimi两轮无可评分成品，分数留空。

In [3]:
index = pd.MultiIndex.from_product([all_systems, [1, 2]], names=["AI", "轮次"])
counts = detail.groupby(["AI", "轮次"]).size().reindex(index, fill_value=0)
assert counts.loc[("Claude", 2)] == 0
assert counts.loc["Kimi"].sum() == 0
assert (counts[counts.gt(0)] == 25).all()
display(counts.rename("已评分项数").unstack())

轮次,1,2
AI,,
Claude,25,0
DeepSeek,25,25
GLM,25,25
GPT,25,25
Gemini,25,25
Grok,25,25
Kimi,0,0
Qwen,25,25


## 3. 评分与用量

S为25项得分之和，满分100。各项满分与档位由现行标准核对；用量单独读取，不换算成S分。

In [4]:
assert len(detail) == 325
dimension = detail.groupby(["AI", "轮次", "维度"])["得分"].sum().unstack("维度")
maximum = detail.groupby(["AI", "轮次", "维度"])["满分"].sum().unstack("维度")
assert (maximum == pd.Series({"A":15,"B":20,"C":20,"D":25,"E":10,"F":10})).all().all()
totals = dimension.sum(axis=1).reindex(index)
display(totals.rename("S总分").unstack())
metrics = frame("评分与用量")
display(metrics.loc[metrics["指标类别"].str.contains("用量|Token", case=False, na=False), ["AI", "轮次", "指标", "数值", "单位"]].head(16))

轮次,1,2
AI,,
Claude,72.0,NaN
DeepSeek,76.5,87.25
GLM,74.0,81.00
GPT,88.0,98.50
Gemini,39.5,59.50
Grok,84.5,85.25
Kimi,NaN,NaN
Qwen,67.5,75.25


,AI,轮次,指标,数值,单位
0,GPT,第一轮,推理Token（输出子集）,2101.0,Token
1,GPT,第一轮,未缓存输入Token,95370.0,Token
2,GPT,第一轮,缓存写入Token,0.0,Token
3,GPT,第一轮,缓存读取Token,1874432.0,Token
4,GPT,第一轮,输出Token,17224.0,Token
42,DeepSeek,第一轮,未缓存输入Token,311371.0,Token
43,DeepSeek,第一轮,缓存写入Token,0.0,Token
44,DeepSeek,第一轮,缓存读取Token,12838400.0,Token
45,DeepSeek,第一轮,输出Token,95925.0,Token
81,Qwen,第一轮,未缓存输入Token,2022.0,Token


## 4. API计价

按底表保留计价档与费用；目录价供比较，不等于实际账单。

In [5]:
pricing = frame("API计价")
display(pricing[["AI", "轮次", "Token类型", "计价档", "计费Token", "分项费用（元）"]].head(8))

,AI,轮次,Token类型,计价档,计费Token,分项费用（元）
0,GPT,第一轮,未缓存输入,低,95370,6.449682
1,GPT,第一轮,未缓存输入,高,95370,6.449682
2,GPT,第一轮,缓存读取,低,1874432,12.676409
3,GPT,第一轮,缓存读取,高,1874432,12.676409
4,GPT,第一轮,缓存写入,低,0,0.000000
5,GPT,第一轮,缓存写入,高,0,0.000000
6,GPT,第一轮,输出,低,17224,5.824123
7,GPT,第一轮,输出,高,17224,5.824123


## 5. 行为与核验

这里列记录类别和数量；动作次数不代表分析质量，也不换算人工分钟。

In [6]:
for name in ("行为记录", "核验记录"):
    table = frame(name)
    display(table.groupby("动作评审" if name == "行为记录" else "核验类型", dropna=False).size().rename("记录数").to_frame())

,记录数
动作评审,
必要探索,215
直接贡献交付,456
确认无收益,35
管理动作,109


,记录数
核验类型,
交付后修正任务,24
共同主题核查,30
定向前后核查,13
实例观测,45
数值核验,86
文件身份核验,3
窗口敏感性观测,7
结论评审,29
行为片段复核,36


## 6. 工作簿结构

检查工作表和下拉定义。实际Excel操作与报告视觉检查另见最终工作簿验收记录。

In [7]:
assert len(book.sheetnames) == 10
assert sum(len(sheet._charts) for sheet in book) == 8
display(pd.DataFrame({"工作表": book.sheetnames,
                      "行数": [s.max_row for s in book],
                      "图表数": [len(s._charts) for s in book]}))
selectors = {"评分细项": {"B2"}, "分维度比较": {"B3", "F3"}}
for name, expected in selectors.items():
    assert expected.issubset({str(v.sqref) for v in book[name].data_validations.dataValidation})

,工作表,行数,图表数
0,评测总览,80,4
1,评分细项,25,0
2,单AI分析,80,4
3,分维度比较,17,0
4,运行记录,16,0
5,评分与用量,653,0
6,API计价,73,0
7,行为记录,816,0
8,核验记录,295,0
9,字段说明,87,0


## 7. 按评分编号对账

核对Excel与查证数据的325条分值、实际工作与缺口、原件、依据、核查方式和关联问题编号。此项通过表示材料同步，不替代对依据充分性的逐项评审。

In [8]:
print(pipeline.verify_current_scores(scores=scores))
assert detail[["AI", "轮次", "条款"]].duplicated().sum() == 0
for _, run in runs.iterrows():
    rd = {"第一轮":1, "第二轮":2}[run["轮次"]]
    expected = totals.loc[(run["AI"], rd)]
    actual = run["现行 S/100"]
    if pd.isna(expected):
        assert pd.isna(actual) or actual == "", (run["AI"], rd)
    else:
        assert abs(float(actual) - expected) < 1e-9, (run["AI"], rd)
print("运行记录的S总分缓存与325条分项一致。")

{'评分条数': 325, '分值与依据': '逐字段一致', '评分机制': 'S v2.2'}
运行记录的S总分缓存与325条分项一致。


## 8. 六系统两轮配对

每个系统使用自身两轮分数计算变化，再取六系统均值。调查维度的贡献比例以净增分为分母。

In [9]:
paired = totals.unstack().loc[systems].rename(columns={1:"第一轮",2:"第二轮"})
paired["变化"] = paired["第二轮"] - paired["第一轮"]
assert paired.notna().all().all()
display(paired)
means = paired.mean()
def two_places(v):
    return Decimal(str(v)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print(f"六系统：{two_places(means['第一轮'])} → {two_places(means['第二轮'])}，平均提高 {two_places(means['变化'])} 分")
by_round = dimension.loc[systems].groupby("轮次").sum()
delta = by_round.loc[2] - by_round.loc[1]
assert abs(delta.sum() - paired["变化"].sum()) < 1e-9
display(pd.DataFrame({"总增分": delta, "占净增分比例": delta / delta.sum()}).round(4))
print(f"调查维度贡献：{delta['C'] / delta.sum():.1%}")

轮次,第一轮,第二轮,变化
AI,,,
GPT,88.0,98.50,10.50
Grok,84.5,85.25,0.75
DeepSeek,76.5,87.25,10.75
GLM,74.0,81.00,7.00
Qwen,67.5,75.25,7.75
Gemini,39.5,59.50,20.00


六系统：71.67 → 81.13，平均提高 9.46 分


,总增分,占净增分比例
维度,,
A,9.50,0.1674
B,6.25,0.1101
C,28.00,0.4934
D,8.75,0.1542
E,3.25,0.0573
F,1.00,0.0176


调查维度贡献：49.3%


## 9. 各AI六维与总分

保留未评分的空值；六维之和应与同组25项直接求和一致。

In [10]:
summary = dimension.reindex(index)
summary["S"] = summary.sum(axis=1, min_count=6)
direct = detail.groupby(["AI", "轮次"])["得分"].sum().reindex(index)
pd.testing.assert_series_equal(summary["S"], direct, check_names=False)
assert summary.loc[("Claude", 1), "S"] == 72
assert summary.loc["Kimi", "S"].isna().all()
display(summary)
book.close()
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == digest for p, digest in before.items())
print("通过：325条分项和查证依据一致，六维与总分一致，六系统配对；输入文件未改写。")

维度               A      B      C      D      E      F      S
AI       轮次                                                 
GPT      1   14.00  20.00  14.25  19.75  10.00  10.00  88.00
         2   15.00  20.00  20.00  23.50  10.00  10.00  98.50
Grok     1   15.00  19.00  15.75  17.00   8.50   9.25  84.50
         2   13.00  19.00  19.25  17.00   7.75   9.25  85.25
DeepSeek 1   13.25  13.75  17.25  19.00   5.25   8.00  76.50
         2   14.00  17.50  19.25  20.50   6.75   9.25  87.25
GLM      1   12.25  15.25  12.50  18.00   6.75   9.25  74.00
         2   13.25  16.25  17.75  17.00   7.50   9.25  81.00
Qwen     1   10.25  15.25   8.75  18.00   6.75   8.50  67.50
         2   14.25  12.75  14.75  19.00   6.75   7.75  75.25
Gemini   1    5.75   6.50   5.75  10.75   4.25   6.50  39.50
         2   10.50  10.50  11.25  14.25   6.00   7.00  59.50
Claude   1   11.25  15.25  15.75  18.25   6.00   5.50  72.00
         2     NaN    NaN    NaN    NaN    NaN    NaN    NaN
Kimi     1     NaN    NaN    NaN    NaN    NaN    NaN    NaN
         2     NaN    NaN    NaN    NaN    NaN    NaN    NaN

通过：325条分项和查证依据一致，六维与总分一致，六系统配对；输入文件未改写。
